# PHREEQC Surrogate Colab Training Control Panel

Goal: run the same training pipeline on Colab GPU while code stays in GitHub and large data stays in Google Drive.

Model contract for v1:

`Input.txt numeric conditions + time_d sequence -> full Output.txt trajectory`

Expected tensors:

- `X`: `(batch, 301, 20)` = 19 generic conditions plus normalized time
- `Y`: `(batch, 301, 32)` = PHREEQC output features
- Rock name is metadata only. It is not passed into the model input tensor.


## 1. Settings

Edit these paths first. If the zip files are in **Shared with me**, add Drive shortcuts into **MyDrive** and point the paths below to those shortcuts.

In [ ]:
from __future__ import annotations

from pathlib import Path

# Change this to your real GitHub repository URL after the code is pushed.
REPO_URL = "https://github.com/YOUR_GITHUB_USER/YOUR_REPO.git"
BRANCH = "main"
REPO_DIR = Path("/content/chemical_thesis_repo")

# Drive data location. Keep large zip files outside git.
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/phreeqc_data")
CALCITE_ZIP = DRIVE_DATA_DIR / "Calcite_wat_sat_data_3.zip"
DOLOMITE_ZIP = DRIVE_DATA_DIR / "Dolomite_wat_sat_data_2.zip"

# Fast Colab-local working folders.
DATA_ROOT = Path("/content/data")
PROCESSED_ROOT = Path("/content/processed")
RUN_ROOT = Path("/content/runs")
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/phreeqc_surrogate_runs")

RUN_SMOKE_FIRST = True
RUN_SWEEP = False

BASELINE_CONFIG = "configs/conditional_model_v1/full_colab_baseline.yaml"
SWEEP_CONFIGS = [
    "configs/conditional_model_v1/full_colab_lr1e-4.yaml",
    "configs/conditional_model_v1/full_colab_lr3e-4.yaml",
    "configs/conditional_model_v1/full_colab_baseline.yaml",
    "configs/conditional_model_v1/full_colab_lr3e-3.yaml",
    "configs/conditional_model_v1/full_colab_reduce_on_plateau.yaml",
    "configs/conditional_model_v1/full_colab_cosine.yaml",
]


## 2. Mount Drive

This gives Colab access to the zip files and to the output folder where run artifacts will be copied.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Clone or Update the Repo

Colab pulls code from GitHub. After local changes are pushed, rerun this cell to get the latest code.

In [ ]:
import os
import subprocess
import sys

if "YOUR_GITHUB_USER" in REPO_URL:
    raise ValueError("Set REPO_URL to your real GitHub repository before running this cell.")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")


## 4. Install Python Dependencies

This uses the repository `requirements.txt`. Colab already has many packages, but this makes the run reproducible.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


## 5. Copy Data to the Colab VM

We train from `/content/data`, not directly from mounted Drive. This is faster because Drive is slow with many small files.

In [ ]:
import zipfile

DATASETS = [
    (CALCITE_ZIP, DATA_ROOT / "wat_sat" / "Calcite", "Calcite_wat_sat_data_3"),
    (DOLOMITE_ZIP, DATA_ROOT / "wat_sat" / "Dolomite", "Dolomite_wat_sat_data_2"),
]

def extract_dataset(zip_path: Path, target_parent: Path, expected_folder: str) -> Path:
    """Extract one zipped PHREEQC dataset into the path expected by configs."""
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    target_parent.mkdir(parents=True, exist_ok=True)
    expected_path = target_parent / expected_folder
    if expected_path.exists():
        print(f"Already extracted: {expected_path}")
        return expected_path
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(target_parent)
    if not expected_path.exists():
        raise FileNotFoundError(
            f"Expected {expected_path} after extraction. Check the zip internal folder name."
        )
    print(f"Extracted: {expected_path}")
    return expected_path

for zip_path, target_parent, expected_folder in DATASETS:
    extract_dataset(zip_path, target_parent, expected_folder)


## 6. Set Runtime Paths and Check GPU

The YAML configs use environment variables so private Drive paths are not committed to git.

In [ ]:
os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["PROCESSED_ROOT"] = str(PROCESSED_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)

RUN_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 7. Smoke Run

This trains on 4 Calcite runs plus 4 Dolomite runs for 1 epoch. It proves the pipeline works before spending GPU time.

In [ ]:
if RUN_SMOKE_FIRST:
    subprocess.run(
        [sys.executable, "-m", "conditional_model_v1.cli.train", "--config", "configs/conditional_model_v1/smoke_colab.yaml"],
        check=True,
    )


## 8. Full Training or Sweep

Start with the baseline. Set `RUN_SWEEP = True` in the settings cell when you want to compare learning rates and schedulers.

In [ ]:
configs_to_run = SWEEP_CONFIGS if RUN_SWEEP else [BASELINE_CONFIG]

for config_path in configs_to_run:
    print(f"Running {config_path}")
    subprocess.run(
        [sys.executable, "-m", "conditional_model_v1.cli.train", "--config", config_path],
        check=True,
    )


## 9. Copy Run Artifacts Back to Drive

Every run folder contains config, history, metrics, checkpoints, plots, and full prediction arrays.

In [ ]:
import shutil
from datetime import datetime

DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
backup_dir = DRIVE_RUN_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S_colab_runs")
shutil.copytree(RUN_ROOT, backup_dir, dirs_exist_ok=True)
print(f"Copied run artifacts to: {backup_dir}")


## 10. Inspect Latest Run

`feature_metrics.csv` shows all output-feature errors. `eval_predictions.npz` stores every true and predicted trajectory for the eval split.

In [ ]:
import pandas as pd
from IPython.display import Image, display

run_dirs = sorted([path for path in RUN_ROOT.iterdir() if path.is_dir() and path.name[:8].isdigit()])
latest_run = run_dirs[-1]
print("latest run:", latest_run)

display(pd.read_csv(RUN_ROOT / "summary.csv").tail())
display(pd.read_csv(latest_run / "history.csv").tail())
display(pd.read_csv(latest_run / "feature_metrics.csv").sort_values("rmse_original", ascending=False).head(32))

grid_paths = sorted((latest_run / "plots" / "trajectory_examples").glob("*_all_outputs.png"))
for path in grid_paths[:3]:
    print(path.name)
    display(Image(filename=str(path)))

print("full eval arrays:", latest_run / "eval_predictions.npz")


## Key Meanings

- `history.csv`: loss after each epoch.
- `feature_metrics.csv`: one row per PHREEQC output feature.
- `eval_predictions.npz`: full numeric true/predicted trajectories.
- `*_all_outputs.png`: one visual grid showing all output trajectories for one run.
- `plots.max_runs: null`: render PNGs for every evaluated run. This can create many files on the full dataset.
